email agent
- authenticates user
    - only then are they allowed into the "inbox"
    - dynamic tools and prompt on the condition of there being an email and password in state that match hardcoded
- checks "inbox"
    - email in tool
- sends emails
    - human in the loop

In [12]:
from dotenv import load_dotenv

load_dotenv()


True

In [13]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

In [14]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

In [17]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage("Successfully authenticated", tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage("Authentication failed", tool_call_id=runtime.tool_call_id)]
        })

In [18]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

In [19]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = "You are a helpful assistant that can check the inbox and send emails."
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [20]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    "gpt-5-nano",
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )


In [24]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="draft 1")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

In [25]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi Jane! That sounds great. I’m around next week. How about Tuesday at 4 PM or Wednesday morning? If those don’t work, suggest a time. Looking forward to catching up!

— Julie


In [26]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

All set. Draft 1 has been sent.

Would you like me to:
- Set a calendar reminder for the meetup?
- Archive or mark this email as done?
- Sign out of your account for safety?

If you want, I can also draft a follow-up message or help plan the coffee date details.


In [27]:
from pprint import pprint

pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='draft 1', additional_kwargs={}, response_metadata={}, id='0d683c9a-08d7-47c1-8d8e-aa3226f99db9'),
              AIMessage(content='Got it—what would you like “Draft 1” to be? I don’t have the context yet.\n\nPlease tell me:\n- The type of document (e.g., email announcement, project proposal, README, report, policy brief, resume/cover letter, etc.)\n- Topic or purpose\n- Audience\n- Desired tone (formal, friendly, technical, etc.)\n- Length or word count\n- Any key points or sections you want included\n- Deadline (optional)\n\nIf you’re not sure, I can provide quick starter templates for common formats. For example:\n\n- Draft 1: Email announcement\n  - Subject line\n  - Greeting\n  - Message body (what’s changing, why it matters)\n  - Call to action\n  - Sign-off\n\n- Draft 1: Project proposal\n  - Title and executive summary\n  - Problem statement\n  - Proposed solution\n  - Scope and milestones\n  - Budget and resources\n  -